# Train NPE/MNPE From Multiple Configs

Set `CONFIG_PATHS` to the JSON configs to train. Paths are resolved relative to the project root, and the configs are trained sequentially with the same entry point as `scripts/train_npe_from_zarr.py`.


## Settings

In [ ]:
from pathlib import Path

CONFIG_PATHS = [
    Path("configs/inversion.best_response.timeseries.visibility_distance_continuous.json"),
    Path("configs/inversion.best_response.final.visibility_distance_discrete.json"),
]

LOG_LEVEL = "INFO"
STOP_ON_ERROR = True
# Use a text progress bar when the Jupyter frontend cannot render tqdm widgets.
FORCE_TEXT_PROGRESS = True


## Setup

In [ ]:
import logging
import sys
from functools import partial

from IPython.display import JSON, display


def find_project_root(config_paths):
    search_starts = [Path.cwd().resolve()]
    for path in config_paths:
        path = path.expanduser()
        if path.is_absolute():
            search_starts.append(path.resolve().parent)
    for start in search_starts:
        for candidate in (start, *start.parents):
            if (candidate / "scripts" / "train_npe_from_zarr.py").is_file():
                return candidate
    raise RuntimeError(
        "Could not find the project root. Run the notebook from inside the repo "
        "or provide an absolute config path."
    )


PROJECT_ROOT = find_project_root(CONFIG_PATHS)
CONFIG_PATHS = [
    (path if path.is_absolute() else PROJECT_ROOT / path).resolve()
    for path in map(Path.expanduser, CONFIG_PATHS)
]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from network_creation.model_inversion import train_inversion_from_config

log_level = getattr(logging, LOG_LEVEL.upper(), None)
if not isinstance(log_level, int):
    raise ValueError(f"Unknown log level: {LOG_LEVEL!r}")
logging.basicConfig(
    level=log_level,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

if FORCE_TEXT_PROGRESS:
    import tqdm.auto
    from tqdm.std import tqdm as text_tqdm

    tqdm.auto.tqdm = partial(
        text_tqdm,
        file=sys.stdout,
        dynamic_ncols=False,
        mininterval=0.0,
        miniters=1,
        leave=True,
    )

print(f"Project root: {PROJECT_ROOT}")
print("Configs:")
for path in CONFIG_PATHS:
    print(f"  {path}")


## Train

In [ ]:
runs = []

for index, config_path in enumerate(CONFIG_PATHS, start=1):
    print(f"\n=== Run {index}/{len(CONFIG_PATHS)}: {config_path.name} ===")
    try:
        result = train_inversion_from_config(config_path).to_dict()
    except Exception as exc:
        runs.append(
            {
                "config_path": str(config_path),
                "status": "failed",
                "error": repr(exc),
            }
        )
        logging.exception("Training failed for %s", config_path)
        if STOP_ON_ERROR:
            raise
    else:
        runs.append(
            {
                "config_path": str(config_path),
                "status": "ok",
                **result,
            }
        )
        print(f"Saved artifacts to {result['artifact_dir']}")


## Summary

In [ ]:
run_summary = [
    {
        "config_path": run["config_path"],
        "status": run["status"],
        "artifact_dir": run.get("artifact_dir"),
        "tensorboard_dir": run.get("tensorboard_dir"),
        "error": run.get("error"),
    }
    for run in runs
]
display(JSON(run_summary, expanded=True))


## TensorBoard

Run this cell in notebook frontends that support the TensorBoard extension.


In [ ]:
tensorboard_dirs = [
    run["tensorboard_dir"] for run in runs if run["status"] == "ok"
]
tensorboard_dir = tensorboard_dirs[0] if tensorboard_dirs else None

# %load_ext tensorboard
# %tensorboard --logdir $tensorboard_dir
